# Small CNN for the Multimodal Healthcare Triage Assistant

This notebook trains the visual classifier with **QLoRA**.

The base convolution and dense weights are stored as **4-bit NF4** values, with the block scales themselves quantized to 8 bits. Those base weights stay frozen. Training updates only the **LoRA** matrices. When training stops, the LoRA update is merged back into the dequantized weights and saved as `models/visual_cnn.pt`, which is the network `backend.py` loads.

The photographs are generated, saved as PNG files, and loaded back from disk. They are synthetic close-ups, not clinical photos and not a medical device. The network returns a visual flag. A doctor, nurse, or medical officer makes the final decision.

Training stops early when **measured** validation accuracy reaches 95%. The printed accuracy is the number of correct validation images divided by the number of validation images.


## Network

```
Input photograph
    ↓
Resize to 64×64 and divide by 255
    ↓
NF4 conv + LoRA
    ↓
ReLU
    ↓
Max pooling
    ↓
NF4 conv + LoRA
    ↓
ReLU
    ↓
Max pooling
    ↓
NF4 dense + LoRA
    ↓
Visual classification flag
```

Flags:

- `no_obvious_visual_concern`
- `visible_redness_flag`
- `visible_swelling_flag`
- `wound_or_surface_break_flag`
- `unclear_image_flag`


In [1]:
from pathlib import Path
import shutil

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 7
IMAGE_SIZE = 64
RENDER_SIZE = 128
TRAIN_PER_CLASS = 80
VAL_PER_CLASS = 24
BATCH_SIZE = 32
MAX_EPOCHS = 50
EARLY_STOP_VALIDATION_ACCURACY = 0.95
LORA_RANK = 16
LORA_ALPHA = 16
LEARNING_RATE = 1e-3
NF4_BLOCK_SIZE = 64

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

ROOT = Path.cwd().resolve()
if not (ROOT / "backend.py").exists():
    for folder in ROOT.parents:
        if (folder / "backend.py").exists():
            ROOT = folder
            break

MODEL_PATH = ROOT / "models" / "visual_cnn.pt"
PREVIEW_DIR = ROOT / "models" / "previews"
SYNTHETIC_DIR = ROOT / "models" / "synthetic"
for folder in (MODEL_PATH.parent, PREVIEW_DIR, SYNTHETIC_DIR):
    folder.mkdir(parents=True, exist_ok=True)

VISUAL_LABELS = [
    "no_obvious_visual_concern",
    "visible_redness_flag",
    "visible_swelling_flag",
    "wound_or_surface_break_flag",
    "unclear_image_flag",
]

NF4_LEVELS = torch.tensor(
    [
        -1.0,
        -0.6961928009986877,
        -0.5250730514526367,
        -0.39491748809814453,
        -0.28444138169288635,
        -0.18477343022823334,
        -0.09105003625154495,
        0.0,
        0.07958029955625534,
        0.16093020141124725,
        0.24611230194568634,
        0.33791524171829224,
        0.44070982933044434,
        0.5626170039176941,
        0.7229568362236023,
        1.0,
    ],
    dtype=torch.float32,
)

print("Project:", ROOT)
print("Device:", DEVICE)
print("Labels:", VISUAL_LABELS)


Project: C:\Users\Arabinda\OneDrive\Desktop\multimodel--main\multimodel--main
Device: cpu
Labels: ['no_obvious_visual_concern', 'visible_redness_flag', 'visible_swelling_flag', 'wound_or_surface_break_flag', 'unclear_image_flag']


In [2]:
class SmallVisualCNN(nn.Module):
    """Conv, ReLU, pool, Conv, ReLU, pool, dense. This is the network backend.py loads."""

    def __init__(self, num_classes: int = 5):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc = nn.Linear(16 * 16 * 16, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        return self.fc(x)


print(SmallVisualCNN(num_classes=len(VISUAL_LABELS)))


SmallVisualCNN(
  (conv1): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=4096, out_features=5, bias=True)
)


In [3]:
def quantize_nf4(weight: torch.Tensor):
    """Block-wise NF4 with 8-bit double quantization of the block scales."""
    flat = weight.detach().float().reshape(-1).cpu()
    pad = (NF4_BLOCK_SIZE - flat.numel() % NF4_BLOCK_SIZE) % NF4_BLOCK_SIZE
    padded = F.pad(flat, (0, pad))
    blocks = padded.view(-1, NF4_BLOCK_SIZE)
    scales = blocks.abs().amax(dim=1).clamp(min=1e-12)
    scale_max = scales.max().clamp(min=1e-12)
    scale_codes = torch.round((scales / scale_max) * 255).clamp(0, 255).to(torch.uint8)
    normalized = blocks / scales.unsqueeze(1)
    codes = (normalized.unsqueeze(-1) - NF4_LEVELS).abs().argmin(dim=-1).to(torch.uint8)
    return codes, scale_codes, scale_max, tuple(weight.shape), pad


class QLoRAConv2d(nn.Module):
    """Frozen NF4 convolution plus a trainable LoRA update."""

    def __init__(self, conv: nn.Conv2d, rank: int, alpha: float):
        super().__init__()
        codes, scale_codes, scale_max, shape, pad = quantize_nf4(conv.weight)
        self.register_buffer("nf4_levels", NF4_LEVELS.clone())
        self.register_buffer("codes", codes)
        self.register_buffer("scale_codes", scale_codes)
        self.register_buffer("scale_max", scale_max.reshape(()))
        self.shape = shape
        self.pad = pad
        self.stride = conv.stride
        self.padding = conv.padding
        self.dilation = conv.dilation
        self.groups = conv.groups
        self.bias = nn.Parameter(conv.bias.detach().float().clone(), requires_grad=False)
        out_channels, in_channels, kernel_h, kernel_w = shape
        self.rank = rank
        self.lora_scale = alpha / rank
        self.lora_A = nn.Parameter(torch.empty(rank, in_channels, kernel_h, kernel_w))
        self.lora_B = nn.Parameter(torch.zeros(out_channels, rank, 1, 1))
        nn.init.kaiming_uniform_(self.lora_A, a=5 ** 0.5)

    def base_weight(self) -> torch.Tensor:
        scales = self.scale_codes.float() / 255.0 * self.scale_max
        values = self.nf4_levels[self.codes.long()] * scales.unsqueeze(1)
        flat = values.reshape(-1)
        if self.pad:
            flat = flat[:-self.pad]
        return flat.reshape(self.shape).to(dtype=self.lora_A.dtype)

    def effective_weight(self) -> torch.Tensor:
        delta = torch.einsum("or,rihw->oihw", self.lora_B[:, :, 0, 0], self.lora_A)
        return self.base_weight() + self.lora_scale * delta

    def forward(self, x):
        return F.conv2d(
            x,
            self.effective_weight(),
            self.bias,
            self.stride,
            self.padding,
            self.dilation,
            self.groups,
        )


class QLoRALinear(nn.Module):
    """Frozen NF4 linear layer plus a trainable LoRA update."""

    def __init__(self, linear: nn.Linear, rank: int, alpha: float):
        super().__init__()
        codes, scale_codes, scale_max, shape, pad = quantize_nf4(linear.weight)
        self.register_buffer("nf4_levels", NF4_LEVELS.clone())
        self.register_buffer("codes", codes)
        self.register_buffer("scale_codes", scale_codes)
        self.register_buffer("scale_max", scale_max.reshape(()))
        self.shape = shape
        self.pad = pad
        self.bias = nn.Parameter(linear.bias.detach().float().clone(), requires_grad=False)
        self.lora_scale = alpha / rank
        self.lora_A = nn.Parameter(torch.empty(rank, linear.in_features))
        self.lora_B = nn.Parameter(torch.zeros(linear.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=5 ** 0.5)

    def base_weight(self) -> torch.Tensor:
        scales = self.scale_codes.float() / 255.0 * self.scale_max
        values = self.nf4_levels[self.codes.long()] * scales.unsqueeze(1)
        flat = values.reshape(-1)
        if self.pad:
            flat = flat[:-self.pad]
        return flat.reshape(self.shape).to(dtype=self.lora_A.dtype)

    def effective_weight(self) -> torch.Tensor:
        return self.base_weight() + self.lora_scale * (self.lora_B @ self.lora_A)

    def forward(self, x):
        return F.linear(x, self.effective_weight(), self.bias)


class QLoRAVisualCNN(nn.Module):
    """Same forward layout as SmallVisualCNN. Only the LoRA matrices train."""

    def __init__(self, num_classes: int, rank: int, alpha: float):
        super().__init__()
        base = SmallVisualCNN(num_classes)
        self.conv1 = QLoRAConv2d(base.conv1, rank, alpha)
        self.conv2 = QLoRAConv2d(base.conv2, rank, alpha)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc = QLoRALinear(base.fc, rank, alpha)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        return self.fc(x)

    def merged_network(self) -> SmallVisualCNN:
        merged = SmallVisualCNN(self.fc.lora_B.shape[0])
        merged.conv1.weight.data.copy_(self.conv1.effective_weight().detach())
        merged.conv1.bias.data.copy_(self.conv1.bias.detach())
        merged.conv2.weight.data.copy_(self.conv2.effective_weight().detach())
        merged.conv2.bias.data.copy_(self.conv2.bias.detach())
        merged.fc.weight.data.copy_(self.fc.effective_weight().detach())
        merged.fc.bias.data.copy_(self.fc.bias.detach())
        return merged


model = QLoRAVisualCNN(len(VISUAL_LABELS), LORA_RANK, LORA_ALPHA).to(DEVICE)
trainable = [(name, param.numel()) for name, param in model.named_parameters() if param.requires_grad]
frozen = [(name, param.numel()) for name, param in model.named_parameters() if not param.requires_grad]
if not trainable or any("lora_" not in name for name, _ in trainable):
    raise RuntimeError("QLoRA setup failed: the optimizer would update more than the LoRA matrices.")
print("Trainable LoRA parameters:")
for name, count in trainable:
    print(f"  {name}: {count}")
nf4_weights = (
    int(torch.tensor(model.conv1.shape).prod())
    + int(torch.tensor(model.conv2.shape).prod())
    + int(torch.tensor(model.fc.shape).prod())
)
print("Trainable total:", sum(count for _, count in trainable))
print("Frozen NF4 base weights:", nf4_weights)
print("Frozen biases:", sum(count for _, count in frozen))


Trainable LoRA parameters:
  conv1.lora_A: 432
  conv1.lora_B: 128
  conv2.lora_A: 1152
  conv2.lora_B: 256
  fc.lora_A: 65536
  fc.lora_B: 80
Trainable total: 67584
Frozen NF4 base weights: 21848
Frozen biases: 29


## Synthetic photographs

Each image is rendered with skin color, pores, blotches, and a light falloff, then saved under `models/synthetic/`. Training reads those files.

- No obvious concern: skin only
- Redness: soft red patches
- Swelling: a shaded raised center
- Wound or surface break: a dark irregular line with a red edge
- Unclear: one of the above, then blurred, darkened, and noised


In [4]:
def lattice_noise(rng, size, cells):
    grid = rng.random((cells, cells)).astype(np.float32)
    picture = Image.fromarray(grid, mode="F").resize((size, size), Image.Resampling.BICUBIC)
    return np.asarray(picture, dtype=np.float32)


def skin_photo(rng, size):
    melanin = float(rng.uniform(0.12, 0.82))
    hemoglobin = float(rng.uniform(0.05, 0.35))
    base = np.array(
        [
            0.93 - 0.48 * melanin + 0.08 * hemoglobin,
            0.76 - 0.46 * melanin,
            0.66 - 0.48 * melanin - 0.05 * hemoglobin,
        ],
        dtype=np.float32,
    )
    blotch = lattice_noise(rng, size, 8)
    pores = lattice_noise(rng, size, size // 3)
    fine = rng.normal(0, 0.015, size=(size, size, 1)).astype(np.float32)
    image = base * (0.82 + 0.22 * blotch)[..., None]
    image = image + (pores[..., None] - 0.5) * 0.05 + fine
    yy, xx = np.mgrid[0:size, 0:size]
    light_y = float(rng.normal(size * 0.35, size * 0.08))
    light_x = float(rng.normal(size * 0.35, size * 0.08))
    falloff = np.sqrt((yy - light_y) ** 2 + (xx - light_x) ** 2) / size
    shade = np.clip(1.08 - 0.45 * falloff, 0.55, 1.15).astype(np.float32)
    return np.clip(image * shade[..., None], 0, 1)


def add_redness(image, rng):
    size = image.shape[0]
    yy, xx = np.mgrid[0:size, 0:size]
    field = lattice_noise(rng, size, 7)
    mask = np.zeros((size, size), dtype=np.float32)
    for _ in range(int(rng.integers(3, 6))):
        cy = int(rng.integers(size // 5, 4 * size // 5))
        cx = int(rng.integers(size // 5, 4 * size // 5))
        radius = float(rng.uniform(size * 0.18, size * 0.40))
        dist = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
        spot = np.clip(1 - dist / radius, 0, 1) ** 2
        mask = np.maximum(mask, spot * (0.70 + 0.30 * field))
    red = np.array([0.90, 0.10, 0.08], dtype=np.float32)
    strength = np.clip(mask[..., None] * float(rng.uniform(0.88, 1.0)), 0, 1)
    return np.clip(image * (1 - strength) + red * strength, 0, 1)


def add_swelling(image, rng):
    size = image.shape[0]
    yy, xx = np.mgrid[0:size, 0:size]
    cy = int(rng.integers(size // 4, 3 * size // 4))
    cx = int(rng.integers(size // 4, 3 * size // 4))
    sigma = float(rng.uniform(size * 0.10, size * 0.20))
    height = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma ** 2)).astype(np.float32)
    gy, gx = np.gradient(height)
    shade = np.clip(0.72 + 2.2 * (-0.65 * gx - 0.55 * gy), 0.35, 1.35)
    pale = height[..., None] * np.array([0.10, 0.09, 0.07], dtype=np.float32)
    return np.clip(image * shade[..., None] + pale, 0, 1)


def add_wound(image, rng):
    size = image.shape[0]
    core = np.zeros((size, size), dtype=np.float32)
    y = int(rng.integers(size // 5, 4 * size // 5))
    x = int(rng.integers(size // 10, size // 4))
    steps = int(rng.integers(int(size * 0.7), int(size * 1.3)))
    for _ in range(steps):
        x = int(np.clip(x + int(rng.integers(-1, 4)), 6, size - 7))
        y = int(np.clip(y + int(rng.integers(-3, 4)), 6, size - 7))
        thickness = int(rng.integers(3, 7))
        y0, y1 = max(0, y - thickness), min(size, y + thickness + 1)
        x0, x1 = max(0, x - thickness), min(size, x + thickness + 1)
        core[y0:y1, x0:x1] = 1
    halo = core.copy()
    halo[1:] = np.maximum(halo[1:], core[:-1])
    halo[:-1] = np.maximum(halo[:-1], core[1:])
    halo[:, 1:] = np.maximum(halo[:, 1:], core[:, :-1])
    halo[:, :-1] = np.maximum(halo[:, :-1], core[:, 1:])
    edge = np.clip(halo - core, 0, 1)
    picture = image.copy()
    picture[edge > 0] = picture[edge > 0] * 0.40 + np.array([0.70, 0.12, 0.10], dtype=np.float32) * 0.60
    picture[core > 0] = np.array([0.12, 0.02, 0.02], dtype=np.float32)
    return np.clip(picture, 0, 1)


def box_blur(image):
    padded = np.pad(image, ((1, 1), (1, 1), (0, 0)), mode="edge")
    total = np.zeros_like(image)
    for dy in range(3):
        for dx in range(3):
            total += padded[dy : dy + image.shape[0], dx : dx + image.shape[1]]
    return total / 9.0


def make_unclear(rng, size):
    image = skin_photo(rng, size)
    choice = int(rng.integers(0, 3))
    if choice == 0:
        image = add_redness(image, rng)
    elif choice == 1:
        image = add_swelling(image, rng)
    else:
        image = add_wound(image, rng)
    passes = int(rng.integers(5, 9))
    for _ in range(passes):
        image = box_blur(image)
    exposure = float(rng.uniform(0.12, 0.38))
    noise = rng.normal(0, 0.08, size=image.shape).astype(np.float32)
    return np.clip(image * exposure + noise, 0, 1)


def render_example(label_index, rng):
    if label_index == 4:
        image = make_unclear(rng, RENDER_SIZE)
    else:
        image = skin_photo(rng, RENDER_SIZE)
        if label_index == 1:
            image = add_redness(image, rng)
        elif label_index == 2:
            image = add_swelling(image, rng)
        elif label_index == 3:
            image = add_wound(image, rng)
    picture = Image.fromarray((np.clip(image, 0, 1) * 255).round().astype(np.uint8), mode="RGB")
    return picture.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)


def load_png(path: Path) -> np.ndarray:
    with Image.open(path) as picture:
        rgb = picture.convert("RGB")
        if rgb.size != (IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(f"{path} is {rgb.size}, expected {IMAGE_SIZE}x{IMAGE_SIZE}.")
        return np.asarray(rgb, dtype=np.float32) / 255.0


def write_split(split_name, per_class, seed):
    rng = np.random.default_rng(seed)
    folder = SYNTHETIC_DIR / split_name
    if folder.exists():
        shutil.rmtree(folder)
    for label_index, name in enumerate(VISUAL_LABELS):
        class_dir = folder / name
        class_dir.mkdir(parents=True)
        for index in range(per_class):
            picture = render_example(label_index, rng)
            path = class_dir / f"{index:04d}.png"
            picture.save(path)
            if path.stat().st_size < 200:
                raise RuntimeError(f"Image file is empty: {path}")
            loaded = load_png(path)
            if float(loaded.std()) < 0.02:
                raise RuntimeError(f"Image has no visible detail: {path}")
    print(f"Wrote {per_class * len(VISUAL_LABELS)} photographs to {folder}")


def read_split(split_name):
    folder = SYNTHETIC_DIR / split_name
    records = []
    for label_index, name in enumerate(VISUAL_LABELS):
        paths = sorted((folder / name).glob("*.png"))
        if not paths:
            raise RuntimeError(f"No photographs found in {folder / name}")
        records.extend((path, label_index) for path in paths)
        preview = PREVIEW_DIR / f"{split_name}-{name}.png"
        shutil.copyfile(paths[0], preview)
    return records


write_split("train", TRAIN_PER_CLASS, SEED)
write_split("validation", VAL_PER_CLASS, SEED + 1000)
train_records = read_split("train")
val_records = read_split("validation")
print("Train files", len(train_records), "Validation files", len(val_records))

for label_index, name in enumerate(VISUAL_LABELS):
    pixels = [load_png(path) for path, label in train_records if label == label_index]
    mean = np.mean(np.stack(pixels), axis=(0, 1, 2))
    print(
        f"{name}: {len(pixels)} train files, "
        f"mean RGB {mean[0]:.3f} {mean[1]:.3f} {mean[2]:.3f}"
    )


PermissionError: [WinError 5] Access is denied: 'C:\\Users\\Arabinda\\OneDrive\\Desktop\\multimodel--main\\multimodel--main\\models\\synthetic\\train\\no_obvious_visual_concern'

In [ ]:
class PhotoDataset(Dataset):
    """Reads the saved photographs. Training applies a random horizontal flip."""

    def __init__(self, records, train: bool):
        self.records = list(records)
        self.train = train

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        path, label = self.records[index]
        image = load_png(path)
        tensor = torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1)
        if self.train and torch.rand(1).item() < 0.5:
            tensor = torch.flip(tensor, dims=[2])
        return tensor, int(label)


def run_epoch(split_model, loader, optimizer=None):
    training = optimizer is not None
    split_model.train(training)
    loss_sum = 0.0
    correct = 0
    seen = 0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        if training:
            optimizer.zero_grad()
        logits = split_model(batch_x)
        loss = criterion(logits, batch_y)
        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [param for param in split_model.parameters() if param.requires_grad],
                1.0,
            )
            optimizer.step()
        loss_sum += float(loss.item()) * int(batch_y.shape[0])
        correct += int((logits.argmax(dim=1) == batch_y).sum().item())
        seen += int(batch_y.shape[0])
    if seen == 0:
        raise RuntimeError("The data loader produced no images.")
    return {
        "loss": loss_sum / seen,
        "correct": correct,
        "seen": seen,
        "accuracy": correct / seen,
    }


train_loader = DataLoader(
    PhotoDataset(train_records, train=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    PhotoDataset(val_records, train=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [param for param in model.parameters() if param.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.01,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
)

before = run_epoch(model, val_loader)
print(
    f"Before training: validation accuracy="
    f"{before['correct']}/{before['seen']}={before['accuracy']:.1%}"
)

history = []
best_accuracy = -1.0
best_state = None
stopped_early = False
for epoch in range(1, MAX_EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer)
    val_metrics = run_epoch(model, val_loader)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_correct": train_metrics["correct"],
            "train_seen": train_metrics["seen"],
            "train_accuracy": train_metrics["accuracy"],
            "validation_loss": val_metrics["loss"],
            "validation_correct": val_metrics["correct"],
            "validation_seen": val_metrics["seen"],
            "validation_accuracy": val_metrics["accuracy"],
        }
    )
    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch}: "
        f"lr={current_lr:.6f} "
        f"train loss={train_metrics['loss']:.4f} "
        f"train accuracy={train_metrics['correct']}/{train_metrics['seen']}={train_metrics['accuracy']:.1%} "
        f"validation accuracy={val_metrics['correct']}/{val_metrics['seen']}={val_metrics['accuracy']:.1%}"
    )
    scheduler.step(val_metrics["accuracy"])
    if val_metrics["accuracy"] > best_accuracy:
        best_accuracy = val_metrics["accuracy"]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    if val_metrics["accuracy"] >= EARLY_STOP_VALIDATION_ACCURACY:
        stopped_early = True
        print(
            f"Early stop at epoch {epoch}: measured validation accuracy "
            f"{val_metrics['accuracy']:.1%} reached {EARLY_STOP_VALIDATION_ACCURACY:.0%}."
        )
        break

if best_state is None:
    raise RuntimeError("Training produced no validation score.")
model.load_state_dict(best_state)
merged = model.merged_network().to(DEVICE).eval()
held_out = run_epoch(merged, val_loader)
print(
    "Merged network validation accuracy="
    f"{held_out['correct']}/{held_out['seen']}={held_out['accuracy']:.1%}"
)

checkpoint = {
    "state_dict": {key: value.detach().cpu() for key, value in merged.state_dict().items()},
    "labels": VISUAL_LABELS,
    "image_size": IMAGE_SIZE,
    "training": {
        "method": "QLoRA",
        "quantization": "NF4",
        "double_quantization": "8-bit block scales",
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "epochs_ran": history[-1]["epoch"],
        "max_epochs": MAX_EPOCHS,
        "early_stop_validation_accuracy": EARLY_STOP_VALIDATION_ACCURACY,
        "early_stopped": stopped_early,
        "best_validation_accuracy": held_out["accuracy"],
        "best_validation_correct": held_out["correct"],
        "best_validation_seen": held_out["seen"],
        "history": history,
    },
    "note": (
        "QLoRA adapters were trained on synthetic photographs, then merged into the "
        "CNN. Not a diagnostic model."
    ),
}
torch.save(checkpoint, MODEL_PATH)
print("Saved", MODEL_PATH)
if not stopped_early:
    print(
        "Measured validation accuracy stayed below "
        f"{EARLY_STOP_VALIDATION_ACCURACY:.0%} after {history[-1]['epoch']} epochs. "
        "The file contains that measured score."
    )


In [ ]:
loaded = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
restored = SmallVisualCNN(num_classes=len(VISUAL_LABELS))
restored.load_state_dict(loaded["state_dict"])
restored.eval()

counts = {name: {"correct": 0, "seen": 0} for name in VISUAL_LABELS}
with torch.no_grad():
    for path, label_index in val_records:
        image = load_png(path)
        tensor = torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1).unsqueeze(0)
        probs = torch.softmax(restored(tensor), dim=1)[0]
        predicted = int(torch.argmax(probs).item())
        name = VISUAL_LABELS[label_index]
        counts[name]["seen"] += 1
        counts[name]["correct"] += int(predicted == label_index)

correct = sum(item["correct"] for item in counts.values())
seen = sum(item["seen"] for item in counts.values())
print(f"Checkpoint validation accuracy={correct}/{seen}={correct / seen:.1%}")
for name in VISUAL_LABELS:
    item = counts[name]
    accuracy = item["correct"] / item["seen"]
    print(f"  {name}: {item['correct']}/{item['seen']}={accuracy:.1%}")

print()
print("Sample validation photographs:")
shown = set()
for path, label_index in val_records:
    expected = VISUAL_LABELS[label_index]
    if expected in shown:
        continue
    shown.add(expected)
    image = load_png(path)
    tensor = torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(restored(tensor), dim=1)[0]
    predicted_index = int(torch.argmax(probs).item())
    confidence = float(probs[predicted_index].item())
    flag = VISUAL_LABELS[predicted_index] if confidence >= 0.55 else "unclear_image_flag"
    print(
        f"file={path.name:10} expected={expected:32} "
        f"predicted={flag:32} confidence={confidence:.0%}"
    )

training = loaded["training"]
print()
print(
    f"Training method: {training['method']} {training['quantization']}, "
    f"epochs run: {training['epochs_ran']}, "
    f"early stopped: {training['early_stopped']}"
)
print("This flag is sent to LangChain with the text, voice, and report.")
print("It is not a diagnosis and it does not recommend treatment.")
